# document_list

> Document list table with toolbar and row actions

In [ ]:
#| default_exp components.document_list

In [ ]:
#| export
from typing import Any, List

from fasthtml.common import (
    Div, Span, Table, Thead, Tbody, Tr, Th, Td,
    Button, Input, Label, P
)

# DaisyUI components
from cjm_fasthtml_daisyui.components.actions.button import (
    btn, btn_colors, btn_styles, btn_sizes
)
from cjm_fasthtml_daisyui.components.data_display.table import (
    table, table_modifiers, table_sizes
)
from cjm_fasthtml_daisyui.components.data_input.checkbox import (
    checkbox, checkbox_sizes
)
from cjm_fasthtml_daisyui.utilities.semantic_colors import (
    bg_dui, text_dui, border_dui
)
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, items, justify, gap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Local imports
from cjm_transcript_workflow_management.models import DocumentSummary, ManagementUrls
from cjm_transcript_workflow_management.html_ids import ManagementHtmlIds
from cjm_transcript_workflow_management.utils import format_duration, format_date
from cjm_transcript_workflow_management.components.helpers import (
    render_icon_button, render_media_type_badge, render_empty_state,
    render_delete_modal, render_alert, DEBUG_MANAGEMENT_RENDER
)

## Toolbar

Select All checkbox and bulk action buttons.

In [ ]:
#| export
def render_toolbar(
    urls:ManagementUrls,  # URL bundle for route endpoints
    doc_count:int=0,  # Number of documents in the list
) -> Any:  # Toolbar element
    """Render the document list toolbar with Select All and bulk actions."""
    return Div(
        # Left side: Select All
        Label(
            Input(
                type="checkbox",
                cls=combine_classes(checkbox, checkbox_sizes.sm),
                id=ManagementHtmlIds.SELECT_ALL,
                onclick="toggleSelectAll(this)",
            ),
            Span("Select All", cls=combine_classes(font_size.sm, m.l(2))),
            cls=combine_classes(flex_display, items.center),
        ),
        # Right side: Delete Selected
        Button(
            lucide_icon("trash-2", size=4),
            "Delete Selected",
            cls=combine_classes(
                btn, btn_colors.error, btn_styles.outline, btn_sizes.sm,
                flex_display, items.center, gap(1)
            ),
            id=ManagementHtmlIds.DELETE_SELECTED_BTN,
            disabled=True,
            onclick=f"prepareBulkDeleteModal(); document.getElementById('{ManagementHtmlIds.BULK_DELETE_MODAL}').showModal()",
        ),
        id=ManagementHtmlIds.TOOLBAR,
        cls=combine_classes(flex_display, justify.between, items.center, m.b(2)),
    )

In [ ]:
from fasthtml.common import to_xml
from cjm_transcript_workflow_management.models import ManagementUrls

urls = ManagementUrls(
    list_documents="/manage/documents",
    document_detail="/manage/documents/detail",
    delete_document="/manage/documents/delete",
    delete_selected="/manage/documents/delete-selected",
    export_document="/manage/export/document",
    export_all="/manage/export/all",
    import_graph="/manage/import",
)

toolbar = render_toolbar(urls, doc_count=3)
xml = to_xml(toolbar)
assert ManagementHtmlIds.SELECT_ALL in xml
assert ManagementHtmlIds.DELETE_SELECTED_BTN in xml
assert "Select All" in xml
assert "Delete Selected" in xml
print("Toolbar OK")

Toolbar OK


## Table Row

Single document row with checkbox, info columns, and action buttons.

In [ ]:
#| export
def render_document_row(
    doc:DocumentSummary,  # Document summary data
    urls:ManagementUrls,  # URL bundle for route endpoints
) -> Any:  # Table row element
    """Render a single document row in the list table."""
    doc_id = doc.document_id
    
    return Tr(
        # Checkbox
        Td(Input(
            type="checkbox",
            cls=combine_classes(checkbox, checkbox_sizes.sm),
            name="doc_ids",
            value=doc_id,
            onclick="updateBulkDeleteBtn()",
        )),
        # Title
        Td(doc.title, cls=str(font_weight.medium)),
        # Media type
        Td(render_media_type_badge(doc.media_type)),
        # Segment count
        Td(f"{doc.segment_count} segs"),
        # Duration
        Td(format_duration(doc.total_duration)),
        # Created
        Td(format_date(doc.created_at), cls=str(font_size.sm)),
        # Actions
        Td(
            Div(
                # View detail
                render_icon_button(
                    "eye", "View",
                    size=str(btn_sizes.sm),
                    hx_get=f"{urls.document_detail}/{doc_id}",
                    hx_target=f"#{ManagementHtmlIds.PAGE_CONTENT}",
                    hx_swap="innerHTML",
                ),
                # Export
                render_icon_button(
                    "download", "Export",
                    size=str(btn_sizes.sm),
                    hx_get=f"{urls.export_document}/{doc_id}",
                ),
                # Delete
                render_icon_button(
                    "trash-2", "Delete",
                    color=str(btn_colors.error),
                    size=str(btn_sizes.sm),
                    onclick=(
                        f"prepareDeleteModal('{doc_id}', '{doc.title}', {doc.segment_count}); "
                        f"document.getElementById('{ManagementHtmlIds.DELETE_MODAL}').showModal()"
                    ),
                ),
                cls=combine_classes(flex_display, gap(1)),
            )
        ),
    )

In [ ]:
doc = DocumentSummary(
    document_id="abc-123", title="1. Laying Plans",
    media_type="audio", segment_count=247,
    total_duration=2537.0, created_at=1740000000.0
)
row = render_document_row(doc, urls)
xml = to_xml(row)
assert "1. Laying Plans" in xml
assert "abc-123" in xml
assert "247 segs" in xml
assert "42:17" in xml
print("Row OK")

Row OK


## Document Table

Full table with header row and document rows.

In [ ]:
#| export
def render_document_table(
    documents:List[DocumentSummary],  # List of document summaries
    urls:ManagementUrls,  # URL bundle for route endpoints
) -> Any:  # Table element wrapped in scrollable container
    """Render the document list table."""
    if DEBUG_MANAGEMENT_RENDER:
        print(f"[RENDER] document_table: {len(documents)} docs")
    
    return Div(
        Table(
            Thead(Tr(
                Th(cls=str(w("10"))),
                Th("Title"),
                Th("Type"),
                Th("Segments"),
                Th("Duration"),
                Th("Created"),
                Th("Actions"),
            )),
            Tbody(
                *[render_document_row(doc, urls) for doc in documents]
            ),
            cls=str(table)
        ),
        cls=combine_classes(
            overflow.x.auto, border_radius.box,
            border(), border_dui.base_content.opacity(5),
            bg_dui.base_100
        )
    )

In [ ]:
docs = [
    DocumentSummary("id-1", "1. Laying Plans", "audio", 247, 2537.0, 1740000000.0),
    DocumentSummary("id-2", "2. Waging War", "audio", 312, 3303.0, 1739900000.0),
]
tbl = render_document_table(docs, urls)
xml = to_xml(tbl)
assert "1. Laying Plans" in xml
assert "2. Waging War" in xml
assert "table" in xml
print("Table OK")

Table OK


## Client-Side JavaScript

Checkbox selection management and delete modal preparation.

In [ ]:
#| export
def render_list_scripts(
    urls:ManagementUrls,  # URL bundle for route endpoints
) -> Any:  # Script element
    """Render client-side JavaScript for checkbox and modal management."""
    from fasthtml.common import Script
    return Script(f"""
    function toggleSelectAll(source) {{
        var checkboxes = document.querySelectorAll('input[name="doc_ids"]');
        checkboxes.forEach(cb => cb.checked = source.checked);
        updateBulkDeleteBtn();
    }}

    function updateBulkDeleteBtn() {{
        var checked = document.querySelectorAll('input[name="doc_ids"]:checked');
        var btn = document.getElementById('{ManagementHtmlIds.DELETE_SELECTED_BTN}');
        if (btn) btn.disabled = checked.length === 0;
    }}

    function prepareDeleteModal(docId, title, segCount) {{
        var body = document.getElementById('{ManagementHtmlIds.DELETE_MODAL_BODY}');
        if (body) {{
            body.innerHTML = '<p>Permanently delete "' + title + '" and all ' + segCount + ' segments?</p>'
                + '<p class="{combine_classes(font_size.sm, text_dui.base_content.opacity(60), m.t(2))}">This action cannot be undone.</p>';
        }}
        var confirmBtn = document.querySelector('#{ManagementHtmlIds.DELETE_MODAL} [data-delete-confirm]');
        if (confirmBtn) {{
            confirmBtn.setAttribute('hx-post', '{urls.delete_document}');
            confirmBtn.setAttribute('hx-vals', JSON.stringify({{doc_id: docId}}));
            confirmBtn.setAttribute('hx-target', '#{ManagementHtmlIds.DOCUMENT_LIST}');
            confirmBtn.setAttribute('hx-swap', 'innerHTML');
            htmx.process(confirmBtn);
        }}
    }}

    function prepareBulkDeleteModal() {{
        var checked = document.querySelectorAll('input[name="doc_ids"]:checked');
        var count = checked.length;
        var body = document.getElementById('{ManagementHtmlIds.BULK_DELETE_MODAL_BODY}');
        if (body) {{
            body.innerHTML = '<p>Permanently delete ' + count + ' document(s) and all their segments?</p>'
                + '<p class="{combine_classes(font_size.sm, text_dui.base_content.opacity(60), m.t(2))}">This action cannot be undone.</p>';
        }}
    }}
    """)

In [ ]:
scripts = render_list_scripts(urls)
xml = to_xml(scripts)
assert "toggleSelectAll" in xml
assert "updateBulkDeleteBtn" in xml
assert "prepareDeleteModal" in xml
print("Scripts OK")

Scripts OK


## Document List

Complete document list component combining toolbar, table, modals, and scripts.

In [ ]:
#| export
def render_document_list(
    documents:List[DocumentSummary],  # List of document summaries
    urls:ManagementUrls,  # URL bundle for route endpoints
) -> Any:  # Complete document list component
    """Render the complete document list with toolbar, table, and modals."""
    if DEBUG_MANAGEMENT_RENDER:
        print(f"[RENDER] document_list: {len(documents)} docs")
    
    # Empty state
    if not documents:
        return Div(
            render_empty_state(),
            id=ManagementHtmlIds.DOCUMENT_LIST,
        )
    
    return Div(
        # Toolbar
        render_toolbar(urls, doc_count=len(documents)),
        # Table
        render_document_table(documents, urls),
        # Alert area for feedback messages
        Div(id=ManagementHtmlIds.ALERT_AREA),
        # Single delete confirmation modal
        render_delete_modal(
            modal_id=ManagementHtmlIds.DELETE_MODAL,
            body_id=ManagementHtmlIds.DELETE_MODAL_BODY,
            title="Delete Document?",
            confirm_attrs={
                "data_delete_confirm": "true",
                "onclick": f"document.getElementById('{ManagementHtmlIds.DELETE_MODAL}').close()",
            },
        ),
        # Bulk delete confirmation modal
        render_delete_modal(
            modal_id=ManagementHtmlIds.BULK_DELETE_MODAL,
            body_id=ManagementHtmlIds.BULK_DELETE_MODAL_BODY,
            title="Delete Selected Documents?",
            confirm_attrs={
                "hx_post": urls.delete_selected,
                "hx_target": f"#{ManagementHtmlIds.DOCUMENT_LIST}",
                "hx_swap": "innerHTML",
                "hx_include": "input[name='doc_ids']",
                "onclick": f"document.getElementById('{ManagementHtmlIds.BULK_DELETE_MODAL}').close()",
            },
        ),
        # Client-side scripts
        render_list_scripts(urls),
        id=ManagementHtmlIds.DOCUMENT_LIST,
    )

In [ ]:
# Test with documents
full_list = render_document_list(docs, urls)
xml = to_xml(full_list)
assert ManagementHtmlIds.DOCUMENT_LIST in xml
assert ManagementHtmlIds.TOOLBAR in xml
assert ManagementHtmlIds.DELETE_MODAL in xml
assert "1. Laying Plans" in xml
print("Full list OK")

Full list OK


In [ ]:
# Test empty state
empty_list = render_document_list([], urls)
xml = to_xml(empty_list)
assert ManagementHtmlIds.DOCUMENT_LIST in xml
assert "No documents found." in xml
print("Empty state OK")

Empty state OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()